# 第12課 - 使用代理記事本進行聊天歷史縮減

本筆記本展示了如何使用 Microsoft 代理框架管理長對話中的上下文。隨著對話增長，令牌數量也會增加——最終超出模型的上下文窗口。我們使用**上下文摘要模式**和可持久化記憶的**代理記事本**來解決此問題。

## 您將學到：
1. **為什麼上下文管理很重要**：理解令牌限制與上下文窗口
2. **上下文感知代理**：構建能管理自身對話上下文的代理
3. **上下文摘要模式**：使用工具濃縮對話歷史
4. **代理記事本**：持久記憶，可超越上下文縮減

## 先決條件：
- 已設定 Azure OpenAI 及環境變數
- 理解前面課程中的基本代理概念


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 為什麼上下文管理很重要

每個 LLM 都有一個有限的**上下文視窗**——它能在一次請求中處理的最大標記數。隨著多輪對話的進行：

- **標記數隨著每則用戶訊息和助理回覆線性增長**。
- **提示標記佔成本主導**，因為整個歷史每輪都會重新發送。
- 最終對話會**超出上下文視窗**，模型要麼截斷，要麼出錯。

### 管理上下文的策略

| 策略 | 工作原理 | 取捨 |
|---|---|---|
| **截斷** | 捨棄最舊的訊息 | 失去早期上下文 |
| **摘要** | 將較舊訊息濃縮成摘要 | 失去部分細節，但保留主要要點 |
| **草稿本 / 外部記憶** | 將關鍵事實儲存在對話外 | 需要調用工具，但可承受任何縮減 |

在此筆記本中，我們將**摘要**與**草稿本工具**結合，讓代理即使在對話歷史被濃縮時也能維持連貫性。


## 創建一個具上下文感知能力的代理


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 模擬一場長時間對話

讓我們演練一段多輪對話，看看上下文如何累積。代理人應該在多輪中保留關鍵細節（偏好、預算、旅行日期）並展現連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

請注意代理人如何保留先前回合的上下文——它知道日本、壽司、寺廟、攝影、3000美元的預算、獨自旅行以及四月的時間範圍。在短時間的對話中，這運作良好，但隨著對話的增長，傳送完整的歷史紀錄變得昂貴。

讓我們繼續進行更多回合的對話，看看上下文的累積：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文摘要模式

隨著對話增長，我們可以使用**摘要工具**將累積的上下文濃縮成緊湊的格式。代理程式呼叫此工具來記錄關鍵偏好，這樣即使舊訊息被刪除，重要資訊仍能被保留。

此模式是更複雜歷史縮減的基礎：
1. 代理程式從對話中識別關鍵事實
2. 呼叫摘要工具來保存這些事實
3. 舊訊息可安全刪除，因為摘要已捕捉重要內容

下面我們定義了一個 `summarize_preferences` 工具，代理程式可呼叫它來記錄所學內容的簡要摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 摘要

在這堂課中，你學會了如何使用 Microsoft Agent Framework 管理長時間執行的代理對話中的上下文：

### 主要概念
- **上下文視窗是有限的** — 對話歷史中的每個代幣都有成本，且計入限制。
- **摘要工具** 讓代理能將累積的上下文濃縮為簡潔摘要，減少代幣用量，同時保留重要資訊。
- **代理草稿板** 提供持久的外部記憶，能夠保存任何對話簡化後的資訊。

### 你所建立的內容
- 一個**具上下文感知能力的代理**，能在多回合對話中維持連貫性
- 一個**摘要工具**（`summarize_preferences`），以簡潔格式記錄關鍵的用戶細節
- 一個**多回合對話**示範上下文保持與變更處理

### 實際應用
- **客服機器人**：在長時間的支援過程中記住偏好設定
- **個人助理**：追蹤進行中的專案，不需重複說明上下文
- **教育導師**：跨多次互動維護學生進度

### 後續步驟
- 實作具檔案持久化功能的完整草稿板工具
- 在摘要後加上自動歷史截斷功能
- 結合向量資料庫以進行語義記憶檢索
- 建立具備幾天後可恢復完整上下文的對話代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於提供準確的翻譯，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
